In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockA (EHF) — 计算 & 缓存 45–75N 区域平均的涡动热通量 v'T'
#
# 功能：
#   1) EXTR: 直接使用 HFLUX (已是某种经向平均的热通量)，
#      对 45–75N 做 cos(lat) 加权平均 → EHF_EXTR(time, lev)
#   2) merged: 使用 BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc 中的
#      V, T 计算 v'T'：
#         v' = V - <V>_lon,  T' = T - <T>_lon
#         v'T' = <v' * T'>_lon
#      再对 45–75N 做 cos(lat) 加权平均 → EHF_MERGED(time, lev)
#
# 输出：
#   EHF_EXTR_45_75N_lev_time.nc   # DataArray: EHF_EXTR(time, lev[Pa])
#   EHF_MERGED_45_75N_lev_time.nc # DataArray: EHF_MERGED(time, lev[Pa])
# ============================================================

import os
import numpy as np
import xarray as xr

# ----- 路径设置 -----
O3_ROOT   = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
EHF_ROOT  = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_EHF"
os.makedirs(EHF_ROOT, exist_ok=True)

EXTR_HFLUX_PATH = (
    "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/"
    "CO2x1SmidEmin_yBWCN.cam.h1.0100-0299_T2Mz_ehflux.isobar.nc"
)

MERGED_INT_PATH = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.002/BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"
)

# 输出 nc
EHF_EXTR_NC   = os.path.join(EHF_ROOT, "EHF_EXTR_45_75N_lev_time.nc")
EHF_MERGED_NC = os.path.join(EHF_ROOT, "EHF_MERGED_45_75N_lev_time.nc")

# 纬度带
LAT_MIN = 45.0
LAT_MAX = 75.0


def area_mean_45_75N(da):
    """
    对给定 DataArray 在 45–75N 做 cos(lat) 加权平均。
    要求 da 至少有 'lat' 维。
    """
    if "lat" not in da.dims:
        raise ValueError(f"area_mean_45_75N: no 'lat' in dims={da.dims}")

    da_band = da.sel(lat=slice(LAT_MIN, LAT_MAX))
    w = np.cos(np.deg2rad(da_band["lat"]))
    return da_band.weighted(w).mean("lat")


def main_blockA_EHF():
    # ================= EXTR HFLUX 部分 =================
    print("[BlockA_EHF] Reading EXTR HFLUX:", EXTR_HFLUX_PATH)
    ds_extr = xr.open_dataset(EXTR_HFLUX_PATH)
    h = ds_extr["HFLUX"]

    # 屏蔽无效值
    h = h.where(np.isfinite(h))

    # 45–75N 区域平均
    ehf_extr_45_75 = area_mean_45_75N(h)  # (time, lev/plev)
    ehf_extr_45_75 = ehf_extr_45_75.transpose("time", "lev")

    # 时间信息
    years_extr = np.unique(ehf_extr_45_75.time.dt.year.values.astype(int))
    n_days = int(ehf_extr_45_75.time.dt.dayofyear.max())
    print("[BlockA_EHF] EXTR years:", years_extr[0], "→", years_extr[-1])
    print("[BlockA_EHF] EXTR n_years:", len(years_extr), "n_days:", n_days)
    print("[BlockA_EHF] EXTR lev:", ehf_extr_45_75["lev"].values)

    # 统一元数据
    ehf_extr_45_75.name = "EHF_EXTR_45_75N"
    ehf_extr_45_75.attrs["long_name"] = "Eddy heat flux (HFLUX) 45–75N"
    # 尽管原文件单位写的是 m/s，这里更接近 K·m/s 的物理量，先标注成 K m s-1
    ehf_extr_45_75.attrs["units"] = "K m s-1"
    ehf_extr_45_75["lev"].attrs["units"] = "hPa"  # HFLUX 这里 lev 是 hPa

    ehf_extr_45_75.to_netcdf(EHF_EXTR_NC)
    ds_extr.close()
    print("[BlockA_EHF] Saved EXTR EHF_45_75N(time,lev) to:", EHF_EXTR_NC)

    # ================= merged V/T → v'T' 部分 =================
    print("[BlockA_EHF] Reading merged INT file:", MERGED_INT_PATH)
    ds_int = xr.open_dataset(MERGED_INT_PATH)
    V = ds_int["V"]  # (time, plev, lat, lon)
    T = ds_int["T"]

    # 屏蔽 fill 值
    V = V.where(np.abs(V) < 1e20)
    T = T.where(np.abs(T) < 1e20)

    print("[BlockA_EHF] V dims:", V.dims, "shape:", V.shape)
    print("[BlockA_EHF] T dims:", T.dims, "shape:", T.shape)

    # 计算 v'、T'（去掉 zonal mean）
    V_zm = V.mean("lon")  # (time, plev, lat)
    T_zm = T.mean("lon")
    V_prime = V - V_zm
    T_prime = T - T_zm

    # v'T' 的经向平均：<v' T'>_lon
    vt_eddy = (V_prime * T_prime).mean("lon")  # (time, plev, lat)
    vt_eddy = vt_eddy.where(np.isfinite(vt_eddy))

    print("[BlockA_EHF] vt_eddy dims:", vt_eddy.dims, "shape:", vt_eddy.shape)

    # 45–75N 区域平均
    vt_eddy_45_75 = area_mean_45_75N(vt_eddy)  # (time, plev)
    vt_eddy_45_75 = vt_eddy_45_75.transpose("time", "plev")

    years_int = np.unique(vt_eddy_45_75.time.dt.year.values.astype(int))
    n_days_int = int(vt_eddy_45_75.time.dt.dayofyear.max())
    print("[BlockA_EHF] merged years:", years_int)
    print("[BlockA_EHF] merged n_days (max DOY):", n_days_int)
    print("[BlockA_EHF] merged plev:", vt_eddy_45_75["plev"].values)

    # 重命名 plev → lev（统一）
    vt_eddy_45_75 = vt_eddy_45_75.rename({"plev": "lev"})
    vt_eddy_45_75["lev"].attrs["units"] = "Pa"

    vt_eddy_45_75.name = "EHF_MERGED_45_75N"
    vt_eddy_45_75.attrs["long_name"] = "Eddy heat flux v'T' 45–75N"
    vt_eddy_45_75.attrs["units"] = "K m s-1"

    vt_eddy_45_75.to_netcdf(EHF_MERGED_NC)
    ds_int.close()
    print("[BlockA_EHF] Saved merged EHF_45_75N(time,lev) to:", EHF_MERGED_NC)

    print("[BlockA_EHF] Done.")


if __name__ == "__main__":
    main_blockA_EHF()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockB (EHF) — 45–75°N, 10 hPa Eddy Heat Flux 折线图
#
# 功能：
#   1) EXTR：10 个低 O3 年 vs 其它年份（45–75N, 10 hPa）
#   2) merged：0008/0013/0014/0019 vs EXTR all-years & low25 气候态
#
# 本版更新：
#   - 明确在图例中标出：
#       * EXTR all-years ±1σ（填色包络）
#       * EXTR climatology (all years)（虚线）
#       * EXTR low-ozone ±1σ（斜线包络）
#       * EXTR low-ozone composite（实线）
#   - 将灰色包络加深 & 提高不透明度，便于查看
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import rcParams

O3_ROOT  = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
EHF_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_EHF"
os.makedirs(EHF_ROOT, exist_ok=True)

EHF_EXTR_NC   = os.path.join(EHF_ROOT, "EHF_EXTR_45_75N_lev_time.nc")
EHF_MERGED_NC = os.path.join(EHF_ROOT, "EHF_MERGED_45_75N_lev_time.nc")
LOW10_TXT     = os.path.join(O3_ROOT, "O3_lowest10_years.txt")
LOW25_TXT     = os.path.join(O3_ROOT, "O3_lowest25pct_years.txt")

N_PREV = 91  # 前一年 Oct–Dec 天数
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)


def get_level_index_pa(lev_vals_pa, target_hpa):
    target_pa = target_hpa * 100.0
    idx = int(np.argmin(np.abs(lev_vals_pa - target_pa)))
    return idx, float(lev_vals_pa[idx] / 100.0)


def build_yearly_composite_matrix(var_da, years_subset, n_days, n_prev=N_PREV):
    """
    构造 Y-1 Oct–Dec + Y Jan–Sep 的按年复合矩阵
    返回 comp_mat: (n_year, 365), years_used
    """
    years_all = var_da.time.dt.year.values.astype(int)
    comp_list = []
    used_years = []

    for y in years_subset:
        if (y not in years_all) or ((y - 1) not in years_all):
            continue

        cur = var_da.sel(time=var_da.time.dt.year == y)
        prev = var_da.sel(time=var_da.time.dt.year == (y - 1))

        if (cur.time.size < n_days) or (prev.time.size < n_days):
            continue

        cur_vals  = np.array(cur)
        prev_vals = np.array(prev)

        series_comp = np.concatenate([
            prev_vals[n_days - n_prev:n_days],
            cur_vals[0:n_days - n_prev],
        ])  # (365,)

        comp_list.append(series_comp)
        used_years.append(y)

    if len(comp_list) == 0:
        return None, np.array([], dtype=int)

    comp_mat = np.stack(comp_list, axis=0)
    return comp_mat, np.array(used_years, dtype=int)


def plot_extremes_vs_others(
    var_extr,
    lowest10_years,
    level_label,
    out_png,
    out_pdf,
):
    """
    图 1：极端 10 年 vs 非极端年
    var_extr: DataArray(time,) — EXTR EHF (45–75N, 单层)
    """
    years_extr = np.unique(var_extr.time.dt.year.values.astype(int))
    n_days = int(var_extr.time.dt.dayofyear.max())
    print(f"[BlockB_EHF] EXTR years for {level_label}:", years_extr)
    print(f"[BlockB_EHF] Days per year:", n_days)

    low_years = np.array(lowest10_years, dtype=int)
    neu_years = years_extr[~np.isin(years_extr, low_years)]

    # 极端年复合
    comp_low, used_low = build_yearly_composite_matrix(
        var_da=var_extr, years_subset=low_years, n_days=n_days, n_prev=N_PREV
    )
    if comp_low is None:
        print("[BlockB_EHF] No valid low-year composite, skip plot.")
        return
    print("[BlockB_EHF] Used low O3 years:", used_low)
    n_extreme = comp_low.shape[0]

    # 非极端年复合
    comp_neu, used_neu = build_yearly_composite_matrix(
        var_da=var_extr, years_subset=neu_years, n_days=n_days, n_prev=N_PREV
    )
    if comp_neu is None:
        print("[BlockB_EHF] No valid neutral-year composite, skip plot.")
        return
    print("[BlockB_EHF] Used neutral years:", used_neu)

    neu_mean = np.nanmean(comp_neu, axis=0)
    neu_std  = np.nanstd(comp_neu, axis=0)

    fig, ax = plt.subplots(1, 1, figsize=(18, 4), constrained_layout=True)

    colors_ext = cm.twilight(np.linspace(0, 1, n_extreme))
    x_full = np.arange(n_days)

    for j in range(n_extreme):
        ax.plot(
            x_full,
            comp_low[j, :],
            color=colors_ext[j],
            alpha=0.8,
            linewidth=2,
            label="low O$_3$ years" if j == 0 else None,
        )

    # === 加深的灰色包络（非极端年 ±1σ） ===
    ax.fill_between(
        x_full,
        neu_mean - neu_std,
        neu_mean + neu_std,
        facecolor="0.6",      # 比 0.8 更深
        alpha=0.5,            # 稍微透明一点但能看清
        edgecolor="0.3",
        linewidth=0.8,
        zorder=0,
        label="all other years ±1σ",
    )
    ax.plot(
        x_full,
        neu_mean,
        color="black",
        linewidth=2.0,
        linestyle="--",
        label="mean of other years",
    )

    ax.set_xlim(0, n_days - 1)
    ax.set_xticks([0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334])
    ax.set_xticklabels(
        ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
         "May", "June", "July", "Aug", "Sep"],
        fontsize=15,
    )

    ax.set_ylabel(f"Eddy heat flux v'T' (45–75°N, {level_label}) [K m s$^{{-1}}$]",
                  fontsize=18)
    ax.tick_params(axis="y", labelsize=16)
    ax.legend(fontsize=14)
    ax.set_title(
        f"Eddy heat flux at 45–75°N, {level_label} — low O$_3$ years vs others",
        fontsize=18,
    )

    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close(fig)

    print("[BlockB_EHF] Saved extreme vs other years figure:")
    print("   ", out_png)
    print("   ", out_pdf)


def plot_special_vs_climatology(
    var_extr,
    var_merged,
    lowest25_years,
    level_label,
    out_png,
    out_pdf,
):
    """
    图 2：0008/13/14/19 vs EXTR all-years & low25 气候态
    var_extr   : EXTR EHF (time,)
    var_merged : merged EHF (time,)
    """
    years_extr = np.unique(var_extr.time.dt.year.values.astype(int))
    n_days = int(var_extr.time.dt.dayofyear.max())
    print(f"[BlockB_EHF] EXTR years ({level_label}):", years_extr)
    print(f"[BlockB_EHF] Days per year:", n_days)

    # ===== ALL-years 复合 =====
    comp_all, used_all = build_yearly_composite_matrix(
        var_da=var_extr,
        years_subset=years_extr,
        n_days=n_days,
        n_prev=N_PREV,
    )
    if comp_all is None:
        print("[BlockB_EHF] No valid ALL-years composite, skip.")
        return
    print("[BlockB_EHF] ALL-years used:", used_all.min(), "→", used_all.max())

    all_comp_mean = np.nanmean(comp_all, axis=0)
    all_comp_std  = np.nanstd(comp_all, axis=0)

    # ===== low25 复合（用 Y-1 + Y 拼接） =====
    low25_years = np.array(lowest25_years, dtype=int)
    comp_low25, used_low25 = build_yearly_composite_matrix(
        var_da=var_extr,
        years_subset=low25_years,
        n_days=n_days,
        n_prev=N_PREV,
    )
    if comp_low25 is None:
        print("[BlockB_EHF] ⚠️ No valid low25 composite, only plot ALL climatology.")
        low25_comp_mean = None
        low25_comp_std  = None
    else:
        print("[BlockB_EHF] low25 years used:", used_low25)
        low25_comp_mean = np.nanmean(comp_low25, axis=0)
        low25_comp_std  = np.nanstd(comp_low25, axis=0)

    # ===== merged special years =====
    years_merged = np.unique(var_merged.time.dt.year.values.astype(int))
    print(f"[BlockB_EHF] merged years ({level_label}):", years_merged)

    missing_special = YEARS_SPECIAL[~np.isin(YEARS_SPECIAL, years_merged)]
    if len(missing_special) > 0:
        print(f"⚠️ Warning: these special years are missing in merged ({level_label}):",
              missing_special)

    rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 10,
        "axes.linewidth": 0.8,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

    fig, ax = plt.subplots(1, 1, figsize=(18, 4.0), constrained_layout=True)
    x_full = np.arange(n_days)
    colors_spec = plt.cm.tab10(np.linspace(0, 1, len(YEARS_SPECIAL)))
    handles_years = []

    # ===== special years 曲线 =====
    for i, y in enumerate(YEARS_SPECIAL):
        if y not in years_merged:
            continue

        cur = var_merged.sel(time=var_merged.time.dt.year == y)
        prev = var_merged.sel(time=var_merged.time.dt.year == (y - 1))

        if (cur.time.size < n_days) or (prev.time.size < n_days):
            print(f"⚠️ Year {y:04d} or {y-1:04d} does not have {n_days} days ({level_label}), skip.")
            continue

        cur_vals  = np.array(cur)
        prev_vals = np.array(prev)

        series_comp = np.concatenate([
            prev_vals[n_days - N_PREV:n_days],
            cur_vals[0:n_days - N_PREV],
        ])

        ax.plot(
            x_full,
            series_comp,
            linestyle="-",
            linewidth=1.5,
            color=colors_spec[i],
            label=f"Year {int(y):04d}",
        )
        handles_years.append(
            Line2D([0], [0], color=colors_spec[i], lw=1.5, label=f"Year {int(y):04d}")
        )

    # ===== ALL-years 包络 + 均值线（颜色加深） =====
    ax.fill_between(
        x_full,
        all_comp_mean - all_comp_std,
        all_comp_mean + all_comp_std,
        facecolor="0.7",   # 比原来的 0.85 深
        alpha=0.8,         # 更实一点
        edgecolor="0.3",
        linewidth=0.8,
        zorder=0,
    )
    ax.plot(
        x_full,
        all_comp_mean,
        linestyle="--",
        linewidth=1.8,
        color="black",
    )

    # ===== low25 包络 + 均值线 =====
    if low25_comp_mean is not None:
        ax.fill_between(
            x_full,
            low25_comp_mean - low25_comp_std,
            low25_comp_mean + low25_comp_std,
            facecolor="none",
            edgecolor="0.3",
            hatch="///",
            linewidth=0.8,
            zorder=1,
        )
        ax.plot(
            x_full,
            low25_comp_mean,
            linestyle="-",
            linewidth=2.0,
            color="black",
        )

    # 轴设置
    ax.set_xlim(0, n_days - 1)
    ax.set_xticks([0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334])
    ax.set_xticklabels(
        ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
         "May", "June", "July", "Aug", "Sep"]
    )
    ax.set_ylabel(f"Eddy heat flux v'T' (45–75°N, {level_label}) [K m s$^{{-1}}$]")

    # ===== 明确图例：年线 + ALL ±1σ + low25 ±1σ =====
    patch_all = Patch(
        facecolor="0.7",
        edgecolor="0.3",
        label="EXTR all-years ±1σ",
    )
    patch_low = Patch(
        facecolor="none",
        edgecolor="0.3",
        hatch="///",
        label="EXTR low-ozone ±1σ",
    )
    line_all = Line2D(
        [0], [0],
        color="black",
        lw=1.8,
        ls="--",
        label="EXTR climatology (all years)",
    )
    line_low = Line2D(
        [0], [0],
        color="black",
        lw=2.0,
        ls="-",
        label="EXTR low-ozone composite",
    )

    handles = handles_years + [line_all, patch_all]
    if low25_comp_mean is not None:
        handles += [line_low, patch_low]

    ax.legend(handles=handles, loc="best", frameon=False, fontsize=8, ncol=2)

    ax.set_title(f"Eddy heat flux (45–75°N, {level_label})", fontsize=11)
    ax.grid(False)

    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close(fig)

    print("[BlockB_EHF] Saved special-years vs climatology figure:")
    print("   ", out_png)
    print("   ", out_pdf)


def main_blockB_EHF():
    ehf_extr = xr.load_dataarray(EHF_EXTR_NC)      # (time, lev[hPa])
    ehf_merged = xr.load_dataarray(EHF_MERGED_NC)  # (time, lev[Pa])

    lev_extr_hpa  = ehf_extr["lev"].values
    lev_merged_pa = ehf_merged["lev"].values

    lowest10_years = np.loadtxt(LOW10_TXT, dtype=int).reshape(-1)
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)
    print("[BlockB_EHF] Lowest 10 O3 years:", lowest10_years)
    print("[BlockB_EHF] Lowest 25% O3 years:", lowest25_years)

    target_levels_hpa = [10.0]

    for target_hpa in target_levels_hpa:
        idx_extr = int(np.argmin(np.abs(lev_extr_hpa - target_hpa)))
        lev_extr_actual = float(lev_extr_hpa[idx_extr])

        idx_merged, lev_merged_actual = get_level_index_pa(lev_merged_pa, target_hpa)

        level_label = f"{lev_extr_actual:.1f} hPa"

        var_extr   = ehf_extr.isel(lev=idx_extr)
        var_merged = ehf_merged.isel(lev=idx_merged)

        print(f"\n[BlockB_EHF] Processing level ~{target_hpa} hPa")
        print("  EXTR index/actual:", idx_extr, lev_extr_actual, "hPa")
        print("  MERGED index/actual:", idx_merged, lev_merged_actual, "Pa")

        out_png1 = os.path.join(
            EHF_ROOT, f"EHF_45_75N_{int(round(lev_extr_actual))}hPa_low10_vs_others.png"
        )
        out_pdf1 = os.path.join(
            EHF_ROOT, f"EHF_45_75N_{int(round(lev_extr_actual))}hPa_low10_vs_others.pdf"
        )
        plot_extremes_vs_others(
            var_extr=var_extr,
            lowest10_years=lowest10_years,
            level_label=level_label,
            out_png=out_png1,
            out_pdf=out_pdf1,
        )

        out_png2 = os.path.join(
            EHF_ROOT,
            f"EHF_45_75N_{int(round(lev_extr_actual))}hPa_special_years_vs_extr_climatology.png"
        )
        out_pdf2 = os.path.join(
            EHF_ROOT,
            f"EHF_45_75N_{int(round(lev_extr_actual))}hPa_special_years_vs_extr_climatology.pdf"
        )
        plot_special_vs_climatology(
            var_extr=var_extr,
            var_merged=var_merged,
            lowest25_years=lowest25_years,
            level_label=level_label,
            out_png=out_png2,
            out_pdf=out_pdf2,
        )

    print("[BlockB_EHF] Done.")


if __name__ == "__main__":
    main_blockB_EHF()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockC (EHF) — 45–75°N v'T' 垂直时间剖面 std.anom + 显著性
#
# 1) 使用 EXTR EHF，计算 200 年 45–75°N HFLUX(time, lev<=850hPa)
#    的日气候态 (mean/std)，并生成基准异常 base_anom_all。
# 2) 使用 INT 文件（全经向），计算 0008/0013/0014/0019 年
#    的 v'T' (eddy heat flux)，45–75°N 平均后，插值到 EXTR 的
#    11 个垂直层 (1, 5, 10, 30, 50, 70, 100, 300, 500, 700, 850 hPa)。
# 3) 对每个 special year 构造 composite 序列：
#      [前一年 Oct–Dec (91 天), 当年 Jan–Sep (274 天)] → 0..364
# 4) 计算 standardized anomaly:
#      std_anom = (ref - clim_mean) / clim_std
# 5) 在“基础异常”上做显著性检验：
#      - t 检验：baseline 日异常是否显著偏离 0，与 ref anomaly 对比
#      - bootstrap：baseline 日异常的均值区间是否包含 ref anomaly
#
# 输出：
#   EHF_vert_std_anom_special_45_75N.nc
#
# coords:
#   ref_year   (ref_year)   [8, 13, 14, 19]
#   lev        (lev)        11 个压力层 (hPa, 1..850)
#   time_index (time_index) 0..364
#   dayofyear  (time_index) [275..365,1..274]
#
# data_vars:
#   std_anom        (ref_year, lev, time_index)     # 标准化异常
#   clim_all_comp   (lev, time_index)              # 气候态 mean
#   clim_std_comp   (lev, time_index)              # 气候态 std
#   t_sig           (ref_year, lev, time_index)    # t 检验显著性
#   b_sig           (ref_year, lev, time_index)    # bootstrap 显著性
# ============================================================

import os
import numpy as np
import xarray as xr
from scipy.stats import t as t_dist

EHF_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_EHF"
os.makedirs(EHF_ROOT, exist_ok=True)

# EXTR EHF 文件（已经是 lon:mean, 包含 HFLUX）
EXTR_EHF_FILE = (
    "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/"
    "CO2x1SmidEmin_yBWCN.cam.h1.0100-0299_T2Mz_ehflux.isobar.nc"
)

# INT 文件：有 U/V/T/O3/Z3 等，带完整 lon，用于计算 v'T'
INT_FILE = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.002/BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"
)

OUT_NC = os.path.join(EHF_ROOT, "EHF_vert_std_anom_special_45_75N.nc")

# composite 设置
N_PREV = 91  # 前一年 Oct–Dec 天数
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)


def bootstrap_ci(data, nboot=5000, alpha=0.05, rng=None):
    """
    对给定样本 data 进行 bootstrap，返回均值的 (low, high) 置信区间。
    """
    data = np.asarray(data)
    data = data[~np.isnan(data)]
    n = data.size
    if n < 2:
        return np.nan, np.nan

    if rng is None:
        rng = np.random.default_rng()

    means = np.empty(nboot)
    for i in range(nboot):
        samp = rng.choice(data, size=n, replace=True)
        means[i] = np.mean(samp)

    low = np.percentile(means, 100 * alpha / 2.0)
    high = np.percentile(means, 100 * (1 - alpha / 2.0))
    return low, high


def compute_significance_for_baseline(
    base_anom,   # xarray.DataArray(time, lev) baseline 日异常
    anom_ref,    # np.ndarray(lev, time_index) 特殊年份的异常（非标准化）
    doy_base,    # baseline 的 DOY 序列 (time,)
    doy_comp,    # composite 的 DOY 序列 (time_index,)
    alpha=0.05,
    nboot=5000,
):
    """
    对每个 (lev, time_index) 计算显著性：
      - t 检验：baseline 日异常是否显著偏离 0（双侧）；以 ref anomaly 为观测。
      - bootstrap：baseline 日异常的 mean 的 bootstrap CI 是否包含 ref anomaly。
    返回：
      t_sig(lev, time_index), b_sig(lev, time_index)  布尔数组
    """
    base_vals = base_anom.values  # (time, lev)
    lev_n = base_vals.shape[1]
    t_len = anom_ref.shape[1]

    t_sig = np.zeros((lev_n, t_len), dtype=bool)
    b_sig = np.zeros((lev_n, t_len), dtype=bool)

    rng = np.random.default_rng(2025)

    for ti in range(t_len):
        day = int(doy_comp[ti])
        mask_day = (doy_base == day)
        if not np.any(mask_day):
            continue

        day_samples = base_vals[mask_day, :]  # (n_samples, lev)

        for li in range(lev_n):
            col = day_samples[:, li]
            col = col[~np.isnan(col)]
            if col.size < 2:
                continue

            obs = anom_ref[li, ti]  # 特定年份该 DOY 的异常（非 std）
            # --- t 检验 ---
            se = np.std(col, ddof=1) / np.sqrt(col.size)
            if se == 0:
                continue
            tstat = obs / se
            pval = 2 * (1 - t_dist.cdf(abs(tstat), df=col.size - 1))
            t_sig[li, ti] = (pval < alpha)

            # --- bootstrap CI ---
            lo, hi = bootstrap_ci(col, nboot=nboot, alpha=alpha, rng=rng)
            if np.isnan(lo) or np.isnan(hi):
                continue
            b_sig[li, ti] = not (lo <= obs <= hi)

    return t_sig, b_sig


def calc_extr_ehf_45_75N():
    """
    从 EXTR EHF 文件中提取 45–75°N HFLUX(time, lev<=850hPa)。
    返回：
      ehf_extr (time, lev)，lev 为 hPa，去掉 990hPa。
    """
    print("[BlockC_EHF] Reading EXTR EHF:", EXTR_EHF_FILE)
    ds_extr = xr.open_dataset(EXTR_EHF_FILE)

    hflux = ds_extr["HFLUX"]  # (time, lev, lat)
    lev_vals = ds_extr["lev"].values  # 单位：level ~ hPa 数字

    # 只保留 <= 850 hPa 的层
    lev_mask = (lev_vals <= 850.0)
    lev_keep = lev_vals[lev_mask]

    # 45–75N 平均
    hflux_45_75 = hflux.sel(lat=slice(45, 75)).mean("lat")  # (time, lev)

    # 选取需要的垂直层
    ehf_extr = hflux_45_75.sel(lev=lev_keep)
    ehf_extr = ehf_extr.transpose("time", "lev")  # (time, lev)

    ds_extr.close()

    print("[BlockC_EHF] EXTR HFLUX 45–75N shape:", ehf_extr.shape)
    print("[BlockC_EHF] lev_keep (hPa):", lev_keep)

    return ehf_extr, lev_keep


def calc_eddy_heat_flux_from_INT():
    """
    从 INT 文件中计算 v'T'：
      - 对每个 special year 及其前一年，计算 v'T'(time, plev, lat, lon)
      - 副产物：返回 VT_45_75_interp[year] = DataArray(time, lev_keep)
        （已经插值到 EXTR 的 lev_keep (hPa) 上）
    """
    print("[BlockC_EHF] Reading INT file:", INT_FILE)
    ds_int = xr.open_dataset(INT_FILE)

    V = ds_int["V"]  # (time, plev, lat, lon) m/s
    T = ds_int["T"]  # (time, plev, lat, lon) K
    plev = ds_int["plev"].values  # Pa

    years_int = ds_int.time.dt.year.values.astype(int)
    all_years_int = np.unique(years_int)
    print("[BlockC_EHF] INT years:", all_years_int)

    vt_by_year = {}  # year -> xr.DataArray(time, plev, lat, lon)

    for y in YEARS_SPECIAL:
        for yy in [y - 1, y]:
            if yy in vt_by_year:
                continue
            if yy not in all_years_int:
                print(f"[BlockC_EHF] ⚠️ Year {yy:04d} not found in INT, skip.")
                continue

            print(f"[BlockC_EHF] Computing v'T' for year {yy:04d} ...")
            V_y = V.sel(time=ds_int.time.dt.year == yy)
            T_y = T.sel(time=ds_int.time.dt.year == yy)

            # 去 zonal mean，计算瘤动 v' 和 T'
            V_zm = V_y.mean("lon")
            T_zm = T_y.mean("lon")

            V_prime = V_y - V_zm
            T_prime = T_y - T_zm

            vt_eddy = (V_prime * T_prime).mean("lon")  # (time, plev, lat)

            vt_by_year[yy] = vt_eddy

    ds_int.close()

    return vt_by_year, plev


def main_blockC_EHF():
    # ===== 1. EXTR 45–75N EHF =====
    ehf_extr, lev_keep = calc_extr_ehf_45_75N()  # (time, lev), lev=hPa <=850
    lev_n = lev_keep.size

    time_extr = ehf_extr["time"]
    years_extr = time_extr.dt.year.values.astype(int)
    doy_extr = time_extr.dt.dayofyear.values.astype(int)
    print("[BlockC_EHF] EXTR years:", np.unique(years_extr))

    n_days = int(time_extr.dt.dayofyear.max())
    print("[BlockC_EHF] Days per year (EXTR):", n_days)

    # ===== 2. EXTR 日气候态 mean/std =====
    clim_all = ehf_extr.groupby("time.dayofyear").mean("time")  # (day, lev)
    clim_std = ehf_extr.groupby("time.dayofyear").std("time")   # (day, lev)

    # baseline anomalies: 每个样本对自己的 DOY 气候态求异常
    clim_all_sel_for_extr = clim_all.sel(dayofyear=time_extr.dt.dayofyear)
    base_anom_all = ehf_extr - clim_all_sel_for_extr  # (time, lev)

    # composite DOY 序列: [Oct(275)..Dec(365), Jan(1)..Sep(274)]
    doy_comp = np.concatenate([
        np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int),
        np.arange(1, n_days - N_PREV + 1, dtype=int),
    ])
    t_len = doy_comp.size
    print("[BlockC_EHF] Composite DOY sequence:", doy_comp[:5], "...", doy_comp[-5:])

    # clim composite (mean/std)，并转成 (lev, time_index)
    clim_all_comp = clim_all.sel(dayofyear=doy_comp)      # (time_index, lev)
    clim_std_comp = clim_std.sel(dayofyear=doy_comp)      # (time_index, lev)

    clim_all_comp_vals = clim_all_comp.transpose("lev", "dayofyear").values  # (lev, t)
    clim_std_comp_vals = clim_std_comp.transpose("lev", "dayofyear").values  # (lev, t)

    # ===== 3. INT v'T' 计算 + 插值到 lev_keep =====
    vt_by_year, plev_int = calc_eddy_heat_flux_from_INT()
    plev_hpa_int = plev_int / 100.0  # Pa → hPa

    # 转成 DataArray 便于插值
    plev_da = xr.DataArray(plev_hpa_int, dims=("plev_hpa",))
    lev_keep_da = xr.DataArray(lev_keep, dims=("lev",))

    # ===== 4. 为 special 年份构造 composite & std.anom & 显著性 =====
    n_ref = len(YEARS_SPECIAL)

    std_anom_arr = np.full((n_ref, lev_n, t_len), np.nan, dtype=float)
    t_sig_arr = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    b_sig_arr = np.zeros((n_ref, lev_n, t_len), dtype=bool)

    for yi, y in enumerate(YEARS_SPECIAL):
        print(f"\n[BlockC_EHF] ==== Special year {y:04d} ====")

        # 检查 y 和 y-1 在 INT v'T' 里都有
        if (y not in vt_by_year) or ((y - 1) not in vt_by_year):
            print(f"[BlockC_EHF] ⚠️ Year {y:04d} or {y-1:04d} V'T' missing, skip.")
            continue

        vt_cur = vt_by_year[y]      # (time, plev, lat)
        vt_prev = vt_by_year[y - 1] # (time, plev, lat)

        # 时间长度检查（期望 365 天）
        if vt_cur.sizes["time"] < n_days or vt_prev.sizes["time"] < n_days:
            print(f"[BlockC_EHF] ⚠️ Year {y:04d} or {y-1:04d} has <{n_days} days, skip.")
            continue

        # 45–75N 平均
        vt_cur_45_75 = vt_cur.sel(lat=slice(45, 75)).mean("lat")   # (time, plev)
        vt_prev_45_75 = vt_prev.sel(lat=slice(45, 75)).mean("lat") # (time, plev)

        # time 维度对应 dayofyear 1..365，视为顺序固定
        vt_cur_lev = vt_cur_45_75.assign_coords(plev_hpa=("plev", plev_hpa_int)) \
                                   .swap_dims({"plev": "plev_hpa"})
        vt_prev_lev = vt_prev_45_75.assign_coords(plev_hpa=("plev", plev_hpa_int)) \
                                    .swap_dims({"plev": "plev_hpa"})

        # 插值到 EXTR 的 lev_keep (hPa)
        vt_cur_interp = vt_cur_lev.interp(plev_hpa=lev_keep_da)   # (time, lev)
        vt_prev_interp = vt_prev_lev.interp(plev_hpa=lev_keep_da) # (time, lev)

        vt_cur_vals = vt_cur_interp.transpose("time", "lev").values   # (365, lev)
        vt_prev_vals = vt_prev_interp.transpose("time", "lev").values # (365, lev)

        # 构造 composite: [前一年 Oct–Dec, 当年 Jan–Sep]
        ref_comp_vals = np.concatenate([
            vt_prev_vals[n_days - N_PREV:n_days, :],   # (91, lev)
            vt_cur_vals[0:n_days - N_PREV, :],         # (274, lev)
        ], axis=0)  # (365, lev)

        ref_comp = ref_comp_vals.T  # (lev, t_len)

        # === 4.1 计算异常（相对于 EXTR clim mean） ===
        anom_ref = ref_comp - clim_all_comp_vals  # (lev, t_len)

        # === 4.2 计算 standardized anomaly ===
        with np.errstate(divide="ignore", invalid="ignore"):
            std_anom = anom_ref / clim_std_comp_vals  # (lev, t_len)
            std_anom[~np.isfinite(std_anom)] = np.nan

        std_anom_arr[yi, :, :] = std_anom

        # === 4.3 显著性检验（基于 baseline EXTR 日异常） ===
        print(f"[BlockC_EHF]  t/boot significance for {y:04d} ...")
        t_sig, b_sig = compute_significance_for_baseline(
            base_anom_all,   # EXTR baseline anomalies (time, lev)
            anom_ref,        # 当前年份的异常（非 std）
            doy_extr,
            doy_comp,
            alpha=0.05,
            nboot=5000,
        )

        t_sig_arr[yi, :, :] = t_sig
        b_sig_arr[yi, :, :] = b_sig

    # ===== 5. 写出 NetCDF =====
    ds_out = xr.Dataset(
        coords={
            "ref_year": ("ref_year", YEARS_SPECIAL),
            "lev": ("lev", lev_keep),  # 单位 hPa
            "time_index": ("time_index", np.arange(t_len)),
            "dayofyear": ("time_index", doy_comp),
        },
        data_vars={
            "std_anom": (("ref_year", "lev", "time_index"), std_anom_arr),
            "clim_all_comp": (("lev", "time_index"), clim_all_comp_vals),
            "clim_std_comp": (("lev", "time_index"), clim_std_comp_vals),
            "t_sig": (("ref_year", "lev", "time_index"), t_sig_arr),
            "b_sig": (("ref_year", "lev", "time_index"), b_sig_arr),
        },
    )

    ds_out["std_anom"].attrs["units"] = "σ"
    ds_out["clim_all_comp"].attrs["units"] = "K m s-1"
    ds_out["clim_std_comp"].attrs["units"] = "K m s-1"
    ds_out["lev"].attrs["units"] = "hPa"

    ds_out.attrs["description"] = (
        "Eddy heat flux v'T' (45–75N) standardized anomalies (σ) "
        "for special years relative to EXTR all-years climatology. "
        "Significance masks from t-test and bootstrap are based on "
        "baseline (EXTR) daily anomalies."
    )

    ds_out.to_netcdf(OUT_NC)
    print(f"[BlockC_EHF] Saved dataset to: {OUT_NC}")
    print("[BlockC_EHF] Done.")


if __name__ == "__main__":
    main_blockC_EHF()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockD (EHF) — 绘制 45–75°N v'T' 垂直时间剖面 std.anom 图
#                   + t-test / bootstrap 显著性区域（分图）
#
# 输入：
#   EHF_vert_std_anom_special_45_75N.nc
#
# 输出（每个 ref_year 两张）：
#   EHF_45_75N_std_anom_refYYYY_tTest.png / .pdf
#   EHF_45_75N_std_anom_refYYYY_boot.png  / .pdf
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches

EHF_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_EHF"
os.makedirs(EHF_ROOT, exist_ok=True)

IN_NC = os.path.join(EHF_ROOT, "EHF_vert_std_anom_special_45_75N.nc")


def plot_one_significance(
    lev_hpa_sorted,
    field_sorted,
    sig_mask,
    ref_year,
    mode,
    mag_threshold=1.0,
):
    """
    画单张图：
      - 背景：std.anom (field_sorted)
      - 显著区域 sig_mask & |σ|>=mag_threshold 用 hatch 标注
      mode: 'tTest' or 'boot'
    """
    n_days = field_sorted.shape[1]
    x = np.arange(n_days)

    fig, ax = plt.subplots(1, 1, figsize=(9, 4), constrained_layout=True)

    # 基础填色图
    vmin, vmax = -4.0, 4.0
    levels = np.linspace(vmin, vmax, 13)

    cf = ax.contourf(
        x,
        lev_hpa_sorted,
        field_sorted,
        levels=levels,
        cmap="RdBu_r",
        extend="both",
    )

    # 振幅阈值
    mag_mask = np.abs(field_sorted) >= mag_threshold
    final_mask = sig_mask & mag_mask  # (lev, time)

    sig_int = final_mask.astype(int)

    # hatch 显著区域
    ax.contourf(
        x,
        lev_hpa_sorted,
        sig_int,
        levels=[0.5, 1.5],
        colors="none",
        hatches=["////"],
        linewidths=0.0,
        alpha=0.0,
        zorder=5,
    )

    # y 轴：log 压力，高空在上
    ax.set_yscale("log")
    ax.set_ylim(850, lev_hpa_sorted.min())

    yticks_all = [1, 5, 10, 30, 50, 70, 100, 200, 300, 500, 700, 850, 1000]
    yticks = [p for p in yticks_all
              if (p >= lev_hpa_sorted.min() and p <= 850)]
    ax.set_yticks(yticks)
    ax.get_yaxis().set_major_formatter(mticker.ScalarFormatter())
    ax.set_ylabel("Pressure (hPa)")

    # x 轴：0..364 → Oct–Sep
    ax.set_xlim(0, n_days - 1)
    ax.set_xticks([0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334])
    ax.set_xticklabels(
        ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
         "May", "Jun", "Jul", "Aug", "Sep"]
    )
    ax.set_xlabel("Time (composite days)")

    cbar = plt.colorbar(cf, ax=ax)
    cbar.set_label("v'T' standardized anomaly (σ)")

    # hatch legend
    label_sig = (
        f"significant by {mode} (p<0.05 & |σ|≥{mag_threshold})"
    )
    hatch_patch = mpatches.Patch(
        facecolor="none",
        edgecolor="k",
        hatch="////",
        label=label_sig,
    )
    ax.legend(handles=[hatch_patch], loc="upper right", fontsize=8, frameon=False)

    ax.set_title(f"Eddy heat flux v'T' std.anom (45–75°N), year {int(ref_year):04d} — {mode}")

    out_png = os.path.join(
        EHF_ROOT, f"EHF_45_75N_std_anom_ref{int(ref_year):04d}_{mode}.png"
    )
    out_pdf = os.path.join(
        EHF_ROOT, f"EHF_45_75N_std_anom_ref{int(ref_year):04d}_{mode}.pdf"
    )
    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close(fig)

    print(f"[BlockD_EHF] Saved {mode} figure for year {int(ref_year):04d} to:")
    print("   ", out_png)
    print("   ", out_pdf)


def main_blockD_EHF():
    ds = xr.open_dataset(IN_NC)

    ref_years = ds["ref_year"].values       # [8, 13, 14, 19]
    lev_hpa = ds["lev"].values             # hPa
    std_anom = ds["std_anom"]              # (ref_year, lev, time_index)
    t_sig = ds["t_sig"]                    # (ref_year, lev, time_index)
    b_sig = ds["b_sig"]

    # 从小到大排序压力层：1..850，后面用 ylim(850, min) 让高空在上
    sort_idx = np.argsort(lev_hpa)
    lev_hpa_sorted = lev_hpa[sort_idx]

    for i, y in enumerate(ref_years):
        field = std_anom.isel(ref_year=i).values  # (lev, time)
        if np.all(~np.isfinite(field)):
            print(f"[BlockD_EHF] Year {int(y):04d} is all NaN, skip.")
            continue

        field_sorted = field[sort_idx, :]  # (lev_sorted, time)

        t_mask = t_sig.isel(ref_year=i).values[sort_idx, :]
        b_mask = b_sig.isel(ref_year=i).values[sort_idx, :]

        # 1) t-test 显著性图
        plot_one_significance(
            lev_hpa_sorted,
            field_sorted,
            t_mask,
            ref_year=y,
            mode="tTest",
            mag_threshold=1.0,
        )

        # 2) bootstrap 显著性图
        plot_one_significance(
            lev_hpa_sorted,
            field_sorted,
            b_mask,
            ref_year=y,
            mode="boot",
            mag_threshold=1.0,
        )

    ds.close()
    print("[BlockD_EHF] Done.")


if __name__ == "__main__":
    main_blockD_EHF()
